# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_11 — Live Paper Trading Dashboard Without Execution

### Research question

How do we monitor a strategy in live paper-trading / shadow mode without sending orders, so that we can detect data failures, signal drift, risk breaches, alpha decay, cost pressure, and live-vs-backtest degradation before production?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
05_05_bayesian_strategy_calibration
05_06_performance_haircut_and_deflated_sharpe
05_07_purged_kfold_and_embargo_cv
05_08_probability_of_backtest_overfitting
05_09_white_reality_check_bootstrap
05_10_turnover_capacity_and_alpha_decay
```

The previous notebooks built validation, cost, overfitting, and capacity controls. This notebook adds the layer that should come **before execution**:

> live paper trading dashboard without sending orders.

No broker connection.  
No real orders.  
No real fills.  
No execution simulation beyond paper accounting.

Instead, this notebook monitors:

1. data freshness;
2. missing and stale prices;
3. signal generation health;
4. target-weight stability;
5. risk exposure;
6. paper PnL;
7. benchmark-relative performance;
8. rolling drawdown;
9. realised volatility;
10. rolling IC;
11. turnover and estimated cost pressure;
12. capacity pressure;
13. live-vs-backtest degradation;
14. distribution drift;
15. regime and stress behaviour;
16. alert generation;
17. dashboard snapshots;
18. go/no-go readiness flags;
19. saved outputs and manifest.

The key idea:

> Shadow mode is where a strategy proves it can run every day, with clean data and stable behaviour, before it is allowed to trade.

## 1. Why paper trading without execution?

A strategy can pass a backtest and still fail operationally.

Common live failures:

- bad data;
- stale prices;
- missing assets;
- signal code breaks;
- feature distributions drift;
- target weights explode;
- turnover is higher than expected;
- capacity assumptions break;
- model stops behaving like the research version;
- benchmark-relative drawdown appears immediately.

Paper trading without execution answers:

> Would we have wanted to trade this today, and would the system have behaved correctly?

It is not proof of profitability.

It is a production-readiness filter.

## 2. What “without execution” means

This notebook deliberately avoids:

- broker API;
- order placement;
- order management;
- fills;
- partial fills;
- execution algorithms;
- live capital movement.

It only computes:

$$
signal_t \rightarrow target\ weights_t \rightarrow paper\ holdings_{t+1}
$$

Paper NAV is based on next-period returns.

This makes the dashboard safe for shadow deployment.

## 3. Dashboard layers

The dashboard has five layers:

### 1. Data health

Are prices, returns, spreads, and volume fresh and valid?

### 2. Signal health

Are signals finite, stable, and within expected ranges?

### 3. Portfolio intent

What would the strategy want to hold?

### 4. Paper performance

How would the target portfolio perform before real execution?

### 5. Governance alerts

Should the strategy remain in research, continue paper trading, or be considered for limited production?

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class PaperTradingDashboardConfig:
    n_research_days: int = 1_500
    n_live_days: int = 180
    annualisation: int = 252
    seed: int = 42
    gross_target: float = 1.20
    max_abs_weight: float = 0.25
    rebalance_step: int = 5
    transaction_cost_bps_estimate: float = 4.0
    max_missing_asset_frac: float = 0.10
    max_stale_days: int = 2
    max_abs_daily_return: float = 0.25
    max_gross_exposure: float = 1.30
    max_net_exposure_abs: float = 0.35
    max_single_name_weight: float = 0.30
    max_daily_turnover: float = 0.75
    max_drawdown_warn: float = -0.08
    max_drawdown_fail: float = -0.15
    min_rolling_20d_sharpe: float = -1.0
    max_live_vol_multiple_vs_research: float = 2.0
    max_signal_drift_z: float = 3.0
    max_cost_drag_ann: float = 0.05
    max_p95_participation: float = 0.10
    paper_aum: float = 25_000_000.0
    cvar_alpha: float = 0.95

config = PaperTradingDashboardConfig()
config

## 5. Simulate research and live market data

We simulate two periods:

1. **Research period** — used to establish baseline expectations.
2. **Live paper period** — used for shadow monitoring.

The live period deliberately includes:

- missing data;
- stale prices;
- spread widening;
- volume drops;
- volatility shifts;
- regime changes.

That makes the dashboard meaningful.

In [ ]:
def simulate_research_and_live_data(config: PaperTradingDashboardConfig):
    rng = np.random.default_rng(config.seed)

    n_total = config.n_research_days + config.n_live_days
    dates = pd.bdate_range("2017-01-01", periods=n_total)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND", "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    adv_usd = pd.Series({
        "US_EQ": 9e9, "EU_EQ": 4e9, "EM_EQ": 1.5e9,
        "US_BOND": 12e9, "EU_BOND": 6e9,
        "GOLD": 3e9, "OIL": 2.5e9, "COPPER": 1e9,
        "FX_CARRY": 2.5e9, "TREND": 0.7e9,
        "REIT": 1.5e9, "BTC_PROXY": 0.8e9,
    })

    base_spread_bps = pd.Series({
        "US_EQ": 1.0, "EU_EQ": 1.5, "EM_EQ": 4.5,
        "US_BOND": 0.7, "EU_BOND": 1.0,
        "GOLD": 2.0, "OIL": 3.5, "COPPER": 3.8,
        "FX_CARRY": 2.8, "TREND": 5.5,
        "REIT": 3.2, "BTC_PROXY": 14.0,
    })

    ann_mu = pd.Series({
        "US_EQ": 0.070, "EU_EQ": 0.060, "EM_EQ": 0.080,
        "US_BOND": 0.025, "EU_BOND": 0.020,
        "GOLD": 0.035, "OIL": 0.045, "COPPER": 0.040,
        "FX_CARRY": 0.030, "TREND": 0.050,
        "REIT": 0.060, "BTC_PROXY": 0.120,
    })

    ann_vol = pd.Series({
        "US_EQ": 0.16, "EU_EQ": 0.18, "EM_EQ": 0.25,
        "US_BOND": 0.06, "EU_BOND": 0.07,
        "GOLD": 0.18, "OIL": 0.32, "COPPER": 0.28,
        "FX_CARRY": 0.10, "TREND": 0.12,
        "REIT": 0.20, "BTC_PROXY": 0.70,
    })

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.006, 0.007, 0.035])
    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.15,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.15,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])
    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    regime = np.zeros(n_total, dtype=int)
    stress_type = np.array(["normal"] * n_total, dtype=object)

    returns = np.zeros((n_total, len(assets)))
    factor_returns = np.zeros((n_total, len(factor_names)))

    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    for t in range(n_total):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        is_live = t >= config.n_research_days
        live_vol_extra = 1.25 if is_live else 1.0

        vol_multiplier = (1.0 if regime[t] == 0 else 2.4) * live_vol_extra
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_multiplier**2)

        u = rng.uniform()
        if u < (0.008 if not is_live else 0.014):
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < (0.014 if not is_live else 0.023):
            stress_type[t] = "liquidity_shock"
            f[0] -= rng.uniform(0.010, 0.040)
            f[2] += rng.uniform(0.010, 0.060)
        elif u < (0.020 if not is_live else 0.030):
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=6, size=len(assets)) * np.sqrt((6 - 2) / 6)
        eps = eps * ann_vol.values / np.sqrt(config.annualisation) * 0.35 * vol_multiplier

        returns[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns = pd.DataFrame(returns, index=dates, columns=assets)
    prices = 100.0 * np.exp(np.log1p(returns).cumsum())

    spread_noise = rng.lognormal(0.0, 0.25, size=returns.shape)
    spread_bps = pd.DataFrame(spread_noise * base_spread_bps.values, index=dates, columns=assets)
    spread_bps = spread_bps.multiply(1.0 + 0.8 * pd.Series(regime, index=dates), axis=0)

    volume_noise = rng.lognormal(0.0, 0.50, size=returns.shape)
    volume_usd = pd.DataFrame(volume_noise * adv_usd.values, index=dates, columns=assets)
    volume_usd = volume_usd.multiply(1.0 - 0.25 * pd.Series(regime, index=dates), axis=0)

    # Inject live data problems.
    live_dates = dates[config.n_research_days:]
    for asset in assets:
        missing_mask = rng.uniform(size=len(live_dates)) < (0.01 if asset != "BTC_PROXY" else 0.03)
        prices.loc[live_dates[missing_mask], asset] = np.nan
        returns.loc[live_dates[missing_mask], asset] = np.nan

    # Stale prices for a few assets on random live days.
    for asset in ["EM_EQ", "TREND", "BTC_PROXY"]:
        stale_starts = rng.choice(len(live_dates) - 3, size=2, replace=False)
        for s in stale_starts:
            stale_days = live_dates[s:s+3]
            prices.loc[stale_days, asset] = prices.loc[live_dates[s-1], asset] if s > 0 else prices.loc[stale_days[0], asset]

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "adv_usd": [adv_usd[a] for a in assets],
        "base_spread_bps": [base_spread_bps[a] for a in assets],
        "ann_mu_assumption": [ann_mu[a] for a in assets],
        "ann_vol_assumption": [ann_vol[a] for a in assets],
    })

    regime_info = pd.DataFrame({
        "regime": regime,
        "stress_type": stress_type,
        "period": ["research"] * config.n_research_days + ["live_paper"] * config.n_live_days,
    }, index=dates)

    factors = pd.DataFrame(factor_returns, index=dates, columns=factor_names)

    benchmark = 0.60 * returns["US_EQ"].fillna(0.0) + 0.25 * returns["US_BOND"].fillna(0.0) + 0.15 * returns["GOLD"].fillna(0.0)

    return returns, prices, spread_bps, volume_usd, metadata, regime_info, factors, benchmark

returns_raw, prices_raw, spread_bps, volume_usd, metadata, regime_info, factors, benchmark = simulate_research_and_live_data(config)
assets = metadata["asset"].tolist()

returns_raw.head(), metadata.head()

## 6. Split research and live paper periods

In [ ]:
research_index = prices_raw.index[:config.n_research_days]
live_index = prices_raw.index[config.n_research_days:]

prices_research_raw = prices_raw.loc[research_index]
prices_live_raw = prices_raw.loc[live_index]

returns_research_raw = returns_raw.loc[research_index]
returns_live_raw = returns_raw.loc[live_index]

pd.Series({
    "research_start": str(research_index[0]),
    "research_end": str(research_index[-1]),
    "live_start": str(live_index[0]),
    "live_end": str(live_index[-1]),
    "n_research_days": len(research_index),
    "n_live_days": len(live_index),
})

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices_raw.index, prices_raw[asset], label=asset, alpha=0.7)
plt.axvline(live_index[0], linestyle="--", label="live paper start")
plt.title("Research + Live Paper Price History")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

## 7. Data health checks

The live dashboard should check:

1. missing prices;
2. stale prices;
3. extreme returns;
4. spread spikes;
5. volume collapses;
6. missing asset coverage.

The strategy should not trade if basic data health fails.

In [ ]:
def compute_returns_from_prices(prices):
    return prices.pct_change()

def data_health_checks(prices, returns, spread_bps, volume_usd, config):
    rows = []

    missing_frac = prices.isna().mean(axis=1)
    stale = prices.diff().abs() < 1e-12
    stale_frac = stale.mean(axis=1)

    extreme_return_frac = (returns.abs() > config.max_abs_daily_return).mean(axis=1)

    spread_z = spread_bps.sub(spread_bps.rolling(63).mean(), axis=0).div(spread_bps.rolling(63).std().replace(0, np.nan), axis=0)
    spread_spike_frac = (spread_z > 4.0).mean(axis=1)

    volume_z = volume_usd.sub(volume_usd.rolling(63).mean(), axis=0).div(volume_usd.rolling(63).std().replace(0, np.nan), axis=0)
    volume_collapse_frac = (volume_z < -3.0).mean(axis=1)

    for date in prices.index:
        rows.append({
            "date": date,
            "missing_asset_frac": missing_frac.loc[date],
            "stale_asset_frac": stale_frac.loc[date],
            "extreme_return_frac": extreme_return_frac.loc[date] if date in returns.index else 0.0,
            "spread_spike_frac": spread_spike_frac.loc[date] if date in spread_spread_index if False else np.nan,
        })

    # Build directly to avoid local name issues.
    health = pd.DataFrame(index=prices.index)
    health["missing_asset_frac"] = missing_frac
    health["stale_asset_frac"] = stale_frac
    health["extreme_return_frac"] = extreme_return_frac.reindex(prices.index).fillna(0.0)
    health["spread_spike_frac"] = spread_spike_frac.reindex(prices.index).fillna(0.0)
    health["volume_collapse_frac"] = volume_collapse_frac.reindex(prices.index).fillna(0.0)

    health["flag_missing_data"] = health["missing_asset_frac"] > config.max_missing_asset_frac
    health["flag_stale_prices"] = health["stale_asset_frac"] > config.max_missing_asset_frac
    health["flag_extreme_returns"] = health["extreme_return_frac"] > 0
    health["flag_spread_spike"] = health["spread_spike_frac"] > 0.25
    health["flag_volume_collapse"] = health["volume_collapse_frac"] > 0.25

    health["data_health_pass"] = ~health[
        [
            "flag_missing_data",
            "flag_stale_prices",
            "flag_extreme_returns",
            "flag_spread_spike",
            "flag_volume_collapse",
        ]
    ].any(axis=1)

    return health

# Cleaned prices for strategy calculation: forward-fill limited, returns from cleaned prices.
prices = prices_raw.ffill(limit=config.max_stale_days)
returns = compute_returns_from_prices(prices).fillna(0.0)

data_health = data_health_checks(prices_raw, returns_raw, spread_bps, volume_usd, config)
data_health_live = data_health.loc[live_index]

data_health_live.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data_health_live.index, data_health_live["missing_asset_frac"], label="missing")
plt.plot(data_health_live.index, data_health_live["stale_asset_frac"], label="stale")
plt.plot(data_health_live.index, data_health_live["spread_spike_frac"], label="spread spike")
plt.plot(data_health_live.index, data_health_live["volume_collapse_frac"], label="volume collapse")
plt.title("Live Paper Data Health Fractions")
plt.xlabel("Date")
plt.ylabel("Fraction of assets")
plt.legend()
plt.show()

## 8. Strategy signal and target weights

This dashboard uses the same signal logic in research and live paper:

- momentum;
- short-term reversal;
- volatility penalty;
- inverse-volatility scaling;
- gross target;
- max weight cap.

In shadow mode, target weights are intents only — not orders.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def build_signal(prices, returns):
    momentum_63 = cross_sectional_zscore(prices.pct_change(63))
    momentum_126 = cross_sectional_zscore(prices.pct_change(126))
    reversal_5 = cross_sectional_zscore(-prices.pct_change(5))
    vol_63 = returns.rolling(63).std() * np.sqrt(config.annualisation)
    low_vol = cross_sectional_zscore(-vol_63)

    signal = 0.45 * momentum_126 + 0.25 * momentum_63 + 0.15 * reversal_5 + 0.15 * low_vol
    return signal.clip(-3, 3).fillna(0.0)

def signal_to_target_weights(signal, returns, config):
    vol = returns.rolling(63).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(config.gross_target).fillna(0.0)

    weights = weights.clip(-config.max_abs_weight, config.max_abs_weight)
    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(config.gross_target).fillna(0.0)
    return weights

def apply_rebalance(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

signal = build_signal(prices, returns)
target_weights_raw = signal_to_target_weights(signal, returns, config)
target_weights = apply_rebalance(target_weights_raw, config.rebalance_step)

signal_live = signal.loc[live_index]
target_weights_live = target_weights.loc[live_index]

target_weights_live.tail()

## 9. Signal health and drift

We compare live signal distributions to research distributions.

For each day:

$$
z_{live} = \frac{stat_{live} - mean(stat_{research})} {std(stat_{research})}
$$

Large z-scores indicate signal drift.

In [ ]:
def signal_health_dashboard(signal, research_index, live_index, config):
    research_signal = signal.loc[research_index]
    live_signal = signal.loc[live_index]

    research_stats = pd.DataFrame({
        "signal_mean": research_signal.mean(axis=1),
        "signal_std": research_signal.std(axis=1),
        "signal_abs_mean": research_signal.abs().mean(axis=1),
        "signal_max_abs": research_signal.abs().max(axis=1),
    })

    live_stats = pd.DataFrame({
        "signal_mean": live_signal.mean(axis=1),
        "signal_std": live_signal.std(axis=1),
        "signal_abs_mean": live_signal.abs().mean(axis=1),
        "signal_max_abs": live_signal.abs().max(axis=1),
    })

    baseline_mean = research_stats.mean()
    baseline_std = research_stats.std().replace(0, np.nan)

    z = live_stats.sub(baseline_mean, axis=1).div(baseline_std, axis=1)

    health = live_stats.copy()
    for col in live_stats.columns:
        health[f"{col}_research_z"] = z[col]

    z_cols = [c for c in health.columns if c.endswith("_research_z")]
    health["max_abs_signal_drift_z"] = health[z_cols].abs().max(axis=1)
    health["flag_signal_drift"] = health["max_abs_signal_drift_z"] > config.max_signal_drift_z
    health["flag_nonfinite_signal"] = ~np.isfinite(live_signal).all(axis=1)

    health["signal_health_pass"] = ~(health["flag_signal_drift"] | health["flag_nonfinite_signal"])
    return health, research_stats, live_stats

signal_health, research_signal_stats, live_signal_stats = signal_health_dashboard(signal, research_index, live_index, config)

signal_health.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(signal_health.index, signal_health["max_abs_signal_drift_z"], label="max abs signal drift z")
plt.axhline(config.max_signal_drift_z, linestyle="--", label="threshold")
plt.title("Live Signal Drift vs Research Baseline")
plt.xlabel("Date")
plt.ylabel("Max absolute z-score")
plt.legend()
plt.show()

## 10. Target-weight and risk monitoring

Risk checks:

- gross exposure;
- net exposure;
- single-name max weight;
- class exposure;
- turnover;
- estimated annual cost drag;
- capacity participation.

In shadow mode, these are proposed positions, not actual positions.

In [ ]:
def target_weight_risk_dashboard(weights, spread_bps, volume_usd, metadata, config):
    held = weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)

    gross = held.abs().sum(axis=1)
    net = held.sum(axis=1)
    max_abs = held.abs().max(axis=1)
    turnover = delta.abs().sum(axis=1)

    estimated_cost = turnover * config.transaction_cost_bps_estimate / 10000.0
    estimated_cost_drag_ann = estimated_cost.rolling(21).mean() * config.annualisation

    participation = delta.abs().mul(config.paper_aum).div(volume_usd.reindex(weights.index).replace(0.0, np.nan))
    p95_participation_rolling = participation.rolling(21).quantile(0.95).max(axis=1)

    risk = pd.DataFrame({
        "gross_exposure": gross,
        "net_exposure": net,
        "max_abs_weight": max_abs,
        "daily_turnover": turnover,
        "estimated_daily_cost": estimated_cost,
        "estimated_cost_drag_ann_21d": estimated_cost_drag_ann,
        "p95_participation_21d": p95_participation_rolling,
    }, index=weights.index)

    risk["flag_gross_exposure"] = risk["gross_exposure"] > config.max_gross_exposure
    risk["flag_net_exposure"] = risk["net_exposure"].abs() > config.max_net_exposure_abs
    risk["flag_single_name_weight"] = risk["max_abs_weight"] > config.max_single_name_weight
    risk["flag_turnover"] = risk["daily_turnover"] > config.max_daily_turnover
    risk["flag_cost_drag"] = risk["estimated_cost_drag_ann_21d"] > config.max_cost_drag_ann
    risk["flag_capacity"] = risk["p95_participation_21d"] > config.max_p95_participation

    risk["risk_health_pass"] = ~risk[
        [
            "flag_gross_exposure",
            "flag_net_exposure",
            "flag_single_name_weight",
            "flag_turnover",
            "flag_cost_drag",
            "flag_capacity",
        ]
    ].any(axis=1)

    return risk, participation

risk_dashboard, participation = target_weight_risk_dashboard(target_weights, spread_bps, volume_usd, metadata, config)
risk_dashboard_live = risk_dashboard.loc[live_index]

risk_dashboard_live.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(risk_dashboard_live.index, risk_dashboard_live["gross_exposure"], label="gross")
plt.plot(risk_dashboard_live.index, risk_dashboard_live["net_exposure"], label="net")
plt.axhline(config.max_gross_exposure, linestyle="--", label="gross limit")
plt.axhline(config.max_net_exposure_abs, linestyle=":", label="net abs limit")
plt.axhline(-config.max_net_exposure_abs, linestyle=":")
plt.title("Live Target Exposure")
plt.xlabel("Date")
plt.ylabel("Exposure")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(risk_dashboard_live.index, risk_dashboard_live["daily_turnover"], label="turnover")
plt.axhline(config.max_daily_turnover, linestyle="--", label="turnover limit")
plt.title("Live Target Turnover")
plt.xlabel("Date")
plt.ylabel("Two-way turnover")
plt.legend()
plt.show()

## 11. Asset-class exposure monitoring

In [ ]:
def class_exposure(weights, metadata):
    class_map = metadata.set_index("asset")["asset_class"]
    rows = []

    for date, row in weights.iterrows():
        for asset_class, assets_in_class in class_map.groupby(class_map).groups.items():
            cols = list(assets_in_class)
            class_weight = row[cols].sum()
            class_gross = row[cols].abs().sum()
            rows.append({
                "date": date,
                "asset_class": asset_class,
                "net_class_exposure": class_weight,
                "gross_class_exposure": class_gross,
            })

    return pd.DataFrame(rows)

class_exposure_table = class_exposure(target_weights_live, metadata)

class_exposure_table.tail()

In [ ]:
pivot_gross_class = class_exposure_table.pivot(index="date", columns="asset_class", values="gross_class_exposure").fillna(0.0)

plt.figure(figsize=(12, 6))
for col in pivot_gross_class.columns:
    plt.plot(pivot_gross_class.index, pivot_gross_class[col], label=col)
plt.title("Live Gross Exposure by Asset Class")
plt.xlabel("Date")
plt.ylabel("Gross exposure")
plt.legend()
plt.show()

## 12. Paper NAV accounting

Paper holdings:

$$
w^{paper}_t = w^{target}_{t-1}
$$

Paper gross return:

$$
R^{paper}_t = \sum_i w^{paper}_{i,t} r_{i,t}
$$

Estimated net return:

$$
R^{net}_t = R^{paper}_t - estimatedCost_t
$$

Again: no execution, no real orders, no fills.

In [ ]:
def run_paper_nav(returns, target_weights, config):
    held = target_weights.shift(1).fillna(0.0)
    delta = held.diff().fillna(held)

    gross_return = (held * returns.reindex_like(held).fillna(0.0)).sum(axis=1)
    turnover = delta.abs().sum(axis=1)
    estimated_cost = turnover * config.transaction_cost_bps_estimate / 10000.0

    net_return = gross_return - estimated_cost
    nav = (1 + net_return).cumprod()

    return pd.DataFrame({
        "gross_return": gross_return,
        "estimated_cost": estimated_cost,
        "net_return": net_return,
        "paper_nav": nav,
        "turnover": turnover,
        "gross_exposure": held.abs().sum(axis=1),
        "net_exposure": held.sum(axis=1),
    }, index=returns.index)

paper_nav = run_paper_nav(returns, target_weights, config)
paper_nav["benchmark_return"] = benchmark.reindex(paper_nav.index).fillna(0.0)
paper_nav["benchmark_nav"] = (1 + paper_nav["benchmark_return"]).cumprod()
paper_nav["active_return"] = paper_nav["net_return"] - paper_nav["benchmark_return"]

paper_nav_live = paper_nav.loc[live_index]

paper_nav_live.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(paper_nav_live.index, paper_nav_live["paper_nav"] / paper_nav_live["paper_nav"].iloc[0], label="paper strategy")
plt.plot(paper_nav_live.index, paper_nav_live["benchmark_nav"] / paper_nav_live["benchmark_nav"].iloc[0], label="benchmark")
plt.title("Live Paper NAV vs Benchmark")
plt.xlabel("Date")
plt.ylabel("Growth of 1 since live start")
plt.legend()
plt.show()

## 13. Performance monitoring

Live paper metrics:

- rolling 20-day return;
- rolling 20-day volatility;
- rolling 20-day Sharpe;
- rolling drawdown;
- active return;
- hit rate;
- CVaR.

Compare live paper behaviour to the research baseline.

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    if len(losses) == 0:
        return np.nan, np.nan
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def performance_snapshot(return_series, nav_series, config):
    r = pd.Series(return_series).dropna()
    if len(r) < 2:
        return {}

    _, mdd = max_drawdown(nav_series.reindex(r.index).dropna())
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    return {
        "n_days": len(r),
        "annualised_return": ann_return,
        "annualised_vol": ann_vol,
        "sharpe_proxy": sharpe,
        "max_drawdown": mdd,
        "hit_rate": (r > 0).mean(),
        "VaR": var,
        "CVaR": cvar,
        "total_return": (1 + r).prod() - 1,
    }

research_performance = performance_snapshot(
    paper_nav.loc[research_index, "net_return"],
    paper_nav.loc[research_index, "paper_nav"],
    config,
)

live_performance = performance_snapshot(
    paper_nav_live["net_return"],
    paper_nav_live["paper_nav"],
    config,
)

performance_comparison = pd.DataFrame([
    {"period": "research_shadow_baseline", **research_performance},
    {"period": "live_paper", **live_performance},
])

performance_comparison

In [ ]:
rolling_20 = pd.DataFrame(index=paper_nav_live.index)
rolling_20["rolling_20d_return"] = paper_nav_live["net_return"].rolling(20).sum()
rolling_20["rolling_20d_vol_ann"] = paper_nav_live["net_return"].rolling(20).std() * np.sqrt(config.annualisation)
rolling_20["rolling_20d_sharpe"] = (
    paper_nav_live["net_return"].rolling(20).mean()
    / paper_nav_live["net_return"].rolling(20).std()
    * np.sqrt(config.annualisation)
)

live_nav_scaled = paper_nav_live["paper_nav"] / paper_nav_live["paper_nav"].iloc[0]
rolling_20["drawdown"] = live_nav_scaled / live_nav_scaled.cummax() - 1.0

rolling_20.tail()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(rolling_20.index, rolling_20["rolling_20d_sharpe"], label="20d rolling Sharpe")
plt.axhline(config.min_rolling_20d_sharpe, linestyle="--", label="warning threshold")
plt.title("Live Paper Rolling 20-Day Sharpe")
plt.xlabel("Date")
plt.ylabel("Sharpe")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(rolling_20.index, rolling_20["drawdown"])
plt.axhline(config.max_drawdown_warn, linestyle="--", label="warning")
plt.axhline(config.max_drawdown_fail, linestyle=":", label="fail")
plt.title("Live Paper Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

## 14. Live-vs-research degradation

Compare live paper metrics with research baseline.

Important degradation checks:

- live Sharpe lower than research;
- live volatility much higher;
- live drawdown worse;
- live cost drag higher;
- live signal drift.

In [ ]:
live_vs_research = pd.DataFrame([{
    "research_sharpe": research_performance["sharpe_proxy"],
    "live_sharpe": live_performance["sharpe_proxy"],
    "sharpe_degradation": live_performance["sharpe_proxy"] - research_performance["sharpe_proxy"],
    "research_vol": research_performance["annualised_vol"],
    "live_vol": live_performance["annualised_vol"],
    "live_vol_multiple": live_performance["annualised_vol"] / research_performance["annualised_vol"] if research_performance["annualised_vol"] > 0 else np.nan,
    "research_drawdown": research_performance["max_drawdown"],
    "live_drawdown": live_performance["max_drawdown"],
    "research_total_return": research_performance["total_return"],
    "live_total_return": live_performance["total_return"],
    "live_estimated_cost_drag_ann": paper_nav_live["estimated_cost"].mean() * config.annualisation,
    "research_estimated_cost_drag_ann": paper_nav.loc[research_index, "estimated_cost"].mean() * config.annualisation,
}])

live_vs_research

## 15. Rolling IC monitoring

The live dashboard monitors whether signals still rank future returns.

Cross-sectional IC:

$$
IC_t = corr(signal_t, r_{t+1})
$$

This is noisy day to day, so we monitor rolling averages.

In [ ]:
def daily_cross_sectional_ic(signal, returns, horizon=1):
    future_returns = returns.shift(-horizon)
    values = []

    for date in signal.index:
        if date not in future_returns.index:
            continue
        s = signal.loc[date]
        r = future_returns.loc[date]
        valid = s.notna() & r.notna()
        if valid.sum() >= 3:
            values.append({"date": date, "ic": s[valid].corr(r[valid])})
    return pd.DataFrame(values).set_index("date")

ic_series = daily_cross_sectional_ic(signal, returns, horizon=1)
ic_series["rolling_20d_ic"] = ic_series["ic"].rolling(20).mean()

ic_research = ic_series.loc[research_index.intersection(ic_series.index)]
ic_live = ic_series.loc[live_index.intersection(ic_series.index)]

ic_monitor = pd.DataFrame([{
    "research_mean_ic": ic_research["ic"].mean(),
    "research_ic_std": ic_research["ic"].std(),
    "live_mean_ic": ic_live["ic"].mean(),
    "live_ic_std": ic_live["ic"].std(),
    "ic_degradation": ic_live["ic"].mean() - ic_research["ic"].mean(),
}])

ic_monitor

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(ic_live.index, ic_live["rolling_20d_ic"], label="live rolling 20d IC")
plt.axhline(ic_research["ic"].mean(), linestyle="--", label="research mean IC")
plt.axhline(0, linestyle=":")
plt.title("Live Rolling IC")
plt.xlabel("Date")
plt.ylabel("IC")
plt.legend()
plt.show()

## 16. Alert engine

Alerts are generated daily.

Severity levels:

- INFO;
- WARN;
- FAIL.

A production dashboard would route these to Slack, email, logs, or an internal monitoring tool.

Here they are stored as a table.

In [ ]:
def build_alerts(data_health, signal_health, risk_dashboard, rolling_20, live_vs_research, ic_monitor, config):
    alerts = []

    for date in data_health.index:
        row = data_health.loc[date]
        if row["flag_missing_data"]:
            alerts.append({"date": date, "severity": "FAIL", "category": "DATA", "message": "Missing asset fraction exceeds threshold."})
        if row["flag_stale_prices"]:
            alerts.append({"date": date, "severity": "WARN", "category": "DATA", "message": "Stale price fraction exceeds threshold."})
        if row["flag_extreme_returns"]:
            alerts.append({"date": date, "severity": "WARN", "category": "DATA", "message": "Extreme return detected."})
        if row["flag_spread_spike"]:
            alerts.append({"date": date, "severity": "WARN", "category": "LIQUIDITY", "message": "Spread spike fraction elevated."})
        if row["flag_volume_collapse"]:
            alerts.append({"date": date, "severity": "WARN", "category": "LIQUIDITY", "message": "Volume collapse fraction elevated."})

    for date in signal_health.index:
        row = signal_health.loc[date]
        if row["flag_signal_drift"]:
            alerts.append({"date": date, "severity": "WARN", "category": "SIGNAL", "message": "Signal distribution drift exceeds threshold."})
        if row["flag_nonfinite_signal"]:
            alerts.append({"date": date, "severity": "FAIL", "category": "SIGNAL", "message": "Non-finite signal detected."})

    for date in risk_dashboard.index:
        row = risk_dashboard.loc[date]
        if row["flag_gross_exposure"]:
            alerts.append({"date": date, "severity": "FAIL", "category": "RISK", "message": "Gross exposure limit breached."})
        if row["flag_net_exposure"]:
            alerts.append({"date": date, "severity": "WARN", "category": "RISK", "message": "Net exposure warning."})
        if row["flag_single_name_weight"]:
            alerts.append({"date": date, "severity": "FAIL", "category": "RISK", "message": "Single-name weight limit breached."})
        if row["flag_turnover"]:
            alerts.append({"date": date, "severity": "WARN", "category": "TURNOVER", "message": "Daily turnover above threshold."})
        if row["flag_cost_drag"]:
            alerts.append({"date": date, "severity": "WARN", "category": "COST", "message": "Estimated annual cost drag above threshold."})
        if row["flag_capacity"]:
            alerts.append({"date": date, "severity": "WARN", "category": "CAPACITY", "message": "Participation above capacity guideline."})

    for date in rolling_20.index:
        row = rolling_20.loc[date]
        if pd.notna(row["drawdown"]) and row["drawdown"] < config.max_drawdown_fail:
            alerts.append({"date": date, "severity": "FAIL", "category": "PERFORMANCE", "message": "Live paper drawdown below fail threshold."})
        elif pd.notna(row["drawdown"]) and row["drawdown"] < config.max_drawdown_warn:
            alerts.append({"date": date, "severity": "WARN", "category": "PERFORMANCE", "message": "Live paper drawdown below warning threshold."})
        if pd.notna(row["rolling_20d_sharpe"]) and row["rolling_20d_sharpe"] < config.min_rolling_20d_sharpe:
            alerts.append({"date": date, "severity": "WARN", "category": "PERFORMANCE", "message": "Rolling 20-day Sharpe below threshold."})

    lvr = live_vs_research.iloc[0]
    if lvr["live_vol_multiple"] > config.max_live_vol_multiple_vs_research:
        alerts.append({"date": risk_dashboard.index[-1], "severity": "WARN", "category": "DRIFT", "message": "Live volatility multiple exceeds research baseline."})

    if ic_monitor.iloc[0]["live_mean_ic"] < 0:
        alerts.append({"date": risk_dashboard.index[-1], "severity": "WARN", "category": "ALPHA", "message": "Live mean IC is negative."})

    alerts_df = pd.DataFrame(alerts)
    if len(alerts_df) == 0:
        alerts_df = pd.DataFrame(columns=["date", "severity", "category", "message"])
    else:
        alerts_df = alerts_df.sort_values(["date", "severity", "category"])

    return alerts_df

alerts = build_alerts(
    data_health_live,
    signal_health,
    risk_dashboard_live,
    rolling_20,
    live_vs_research,
    ic_monitor,
    config,
)

alerts.tail(20)

## 17. Daily dashboard snapshot

The latest dashboard snapshot is what a PM or researcher would review each morning.

In [ ]:
def latest_dashboard_snapshot(
    data_health_live,
    signal_health,
    risk_dashboard_live,
    paper_nav_live,
    rolling_20,
    live_vs_research,
    ic_monitor,
    alerts,
):
    latest = data_health_live.index[-1]

    latest_alerts = alerts[alerts["date"] == latest] if len(alerts) else pd.DataFrame()

    snapshot = {
        "as_of_date": latest,
        "data_health_pass": bool(data_health_live.loc[latest, "data_health_pass"]),
        "signal_health_pass": bool(signal_health.loc[latest, "signal_health_pass"]),
        "risk_health_pass": bool(risk_dashboard_live.loc[latest, "risk_health_pass"]),
        "paper_nav": paper_nav_live.loc[latest, "paper_nav"],
        "paper_total_return_live": paper_nav_live.loc[latest, "paper_nav"] / paper_nav_live["paper_nav"].iloc[0] - 1.0,
        "benchmark_total_return_live": paper_nav_live.loc[latest, "benchmark_nav"] / paper_nav_live["benchmark_nav"].iloc[0] - 1.0,
        "live_drawdown": rolling_20.loc[latest, "drawdown"],
        "rolling_20d_sharpe": rolling_20.loc[latest, "rolling_20d_sharpe"],
        "gross_exposure": risk_dashboard_live.loc[latest, "gross_exposure"],
        "net_exposure": risk_dashboard_live.loc[latest, "net_exposure"],
        "daily_turnover": risk_dashboard_live.loc[latest, "daily_turnover"],
        "estimated_cost_drag_ann_21d": risk_dashboard_live.loc[latest, "estimated_cost_drag_ann_21d"],
        "p95_participation_21d": risk_dashboard_live.loc[latest, "p95_participation_21d"],
        "max_abs_signal_drift_z": signal_health.loc[latest, "max_abs_signal_drift_z"],
        "live_vs_research_sharpe_degradation": live_vs_research.iloc[0]["sharpe_degradation"],
        "live_mean_ic": ic_monitor.iloc[0]["live_mean_ic"],
        "n_alerts_today": len(latest_alerts),
        "n_fail_alerts_today": int((latest_alerts["severity"] == "FAIL").sum()) if len(latest_alerts) else 0,
        "n_warn_alerts_today": int((latest_alerts["severity"] == "WARN").sum()) if len(latest_alerts) else 0,
    }

    return pd.DataFrame([snapshot])

dashboard_snapshot = latest_dashboard_snapshot(
    data_health_live,
    signal_health,
    risk_dashboard_live,
    paper_nav_live,
    rolling_20,
    live_vs_research,
    ic_monitor,
    alerts,
)

dashboard_snapshot.T

## 18. Go / no-go readiness rules

Example readiness tiers:

### RED

Do not trade. Major data, risk, or performance failure.

### AMBER

Continue paper trading. Investigate issues.

### GREEN

Eligible for next review stage, not automatic production.

This notebook does **not** approve trading. It produces evidence for a human review.

In [ ]:
def readiness_decision(dashboard_snapshot, alerts, config):
    snap = dashboard_snapshot.iloc[0]

    fail_alerts = alerts[alerts["severity"] == "FAIL"] if len(alerts) else pd.DataFrame()
    recent_alerts = alerts[alerts["date"] >= alerts["date"].max() - pd.Timedelta(days=30)] if len(alerts) else pd.DataFrame()
    recent_fail_count = int((recent_alerts["severity"] == "FAIL").sum()) if len(recent_alerts) else 0
    recent_warn_count = int((recent_alerts["severity"] == "WARN").sum()) if len(recent_alerts) else 0

    red_conditions = [
        recent_fail_count > 0,
        snap["data_health_pass"] is False,
        snap["signal_health_pass"] is False,
        snap["risk_health_pass"] is False,
        snap["live_drawdown"] < config.max_drawdown_fail,
    ]

    amber_conditions = [
        recent_warn_count > 5,
        snap["live_drawdown"] < config.max_drawdown_warn,
        snap["rolling_20d_sharpe"] < config.min_rolling_20d_sharpe if pd.notna(snap["rolling_20d_sharpe"]) else False,
        snap["live_vs_research_sharpe_degradation"] < -1.0,
        snap["live_mean_ic"] < 0,
    ]

    if any(red_conditions):
        status = "RED_DO_NOT_TRADE"
    elif any(amber_conditions):
        status = "AMBER_CONTINUE_SHADOW_MODE"
    else:
        status = "GREEN_READY_FOR_REVIEW_NOT_APPROVAL"

    decision = pd.DataFrame([{
        "as_of_date": snap["as_of_date"],
        "readiness_status": status,
        "recent_fail_count_30d": recent_fail_count,
        "recent_warn_count_30d": recent_warn_count,
        "requires_human_review": True,
        "can_send_orders": False,
        "note": "This dashboard is explicitly without execution. GREEN means eligible for review, not production approval.",
    }])

    return decision

readiness = readiness_decision(dashboard_snapshot, alerts, config)

readiness

## 19. Alert summary by category

In [ ]:
if len(alerts):
    alert_summary = (
        alerts
        .groupby(["severity", "category"])
        .agg(
            n_alerts=("message", "count"),
            first_date=("date", "min"),
            last_date=("date", "max"),
        )
        .reset_index()
        .sort_values(["severity", "n_alerts"], ascending=[True, False])
    )
else:
    alert_summary = pd.DataFrame(columns=["severity", "category", "n_alerts", "first_date", "last_date"])

alert_summary

In [ ]:
if len(alert_summary):
    plot_df = alert_summary.copy()
    plot_df["label"] = plot_df["severity"] + " / " + plot_df["category"]
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["label"], plot_df["n_alerts"])
    plt.title("Alert Counts by Severity and Category")
    plt.xlabel("Number of alerts")
    plt.ylabel("Alert type")
    plt.show()

## 20. Live paper attribution by asset and class

We attribute paper returns:

$$
Contribution_{i,t} = w_{i,t-1} r_{i,t}
$$

This identifies whether live paper PnL is driven by intended assets or unexpected exposures.

In [ ]:
def paper_attribution(returns, target_weights, metadata, live_index):
    held = target_weights.shift(1).fillna(0.0)
    contribution = held * returns.reindex_like(held).fillna(0.0)
    contribution_live = contribution.loc[live_index]

    rows = []
    for asset in contribution_live.columns:
        rows.append({
            "asset": asset,
            "annualised_contribution": contribution_live[asset].mean() * config.annualisation,
            "total_contribution": contribution_live[asset].sum(),
            "average_weight": held.loc[live_index, asset].mean(),
            "average_abs_weight": held.loc[live_index, asset].abs().mean(),
        })

    asset_attr = pd.DataFrame(rows).merge(metadata[["asset", "asset_class"]], on="asset", how="left")

    class_attr = (
        asset_attr
        .groupby("asset_class")
        .agg(
            annualised_contribution=("annualised_contribution", "sum"),
            total_contribution=("total_contribution", "sum"),
            average_abs_weight=("average_abs_weight", "sum"),
        )
        .reset_index()
        .sort_values("annualised_contribution", ascending=False)
    )

    return asset_attr.sort_values("annualised_contribution", ascending=False), class_attr

asset_attribution_live, class_attribution_live = paper_attribution(returns, target_weights, metadata, live_index)

asset_attribution_live

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = asset_attribution_live.sort_values("annualised_contribution")
plt.barh(plot_df["asset"], plot_df["annualised_contribution"])
plt.axvline(0, linestyle="--")
plt.title("Live Paper Annualised Contribution by Asset")
plt.xlabel("Annualised contribution")
plt.ylabel("Asset")
plt.show()

## 21. Monitoring report card

The report card condenses the dashboard into a small set of review fields.

In [ ]:
monitoring_report_card = pd.DataFrame([{
    "as_of_date": dashboard_snapshot.iloc[0]["as_of_date"],
    "readiness_status": readiness.iloc[0]["readiness_status"],
    "data_health_pass_rate_live": data_health_live["data_health_pass"].mean(),
    "signal_health_pass_rate_live": signal_health["signal_health_pass"].mean(),
    "risk_health_pass_rate_live": risk_dashboard_live["risk_health_pass"].mean(),
    "live_total_return": live_performance["total_return"],
    "live_sharpe": live_performance["sharpe_proxy"],
    "research_sharpe": research_performance["sharpe_proxy"],
    "sharpe_degradation": live_vs_research.iloc[0]["sharpe_degradation"],
    "live_max_drawdown": live_performance["max_drawdown"],
    "live_mean_ic": ic_monitor.iloc[0]["live_mean_ic"],
    "live_ic_degradation": ic_monitor.iloc[0]["ic_degradation"],
    "estimated_cost_drag_ann_live": paper_nav_live["estimated_cost"].mean() * config.annualisation,
    "mean_turnover_live": paper_nav_live["turnover"].mean(),
    "n_alerts_live": len(alerts),
    "n_fail_alerts_live": int((alerts["severity"] == "FAIL").sum()) if len(alerts) else 0,
    "n_warn_alerts_live": int((alerts["severity"] == "WARN").sum()) if len(alerts) else 0,
    "can_send_orders": False,
}])

monitoring_report_card.T

## 22. Governance flags

The governance layer decides whether the paper strategy is stable enough for further review.

It does not approve live trading.

In [ ]:
governance_flags = pd.DataFrame([{
    "as_of_date": dashboard_snapshot.iloc[0]["as_of_date"],
    "readiness_status": readiness.iloc[0]["readiness_status"],
    "data_health_pass_rate_live": data_health_live["data_health_pass"].mean(),
    "signal_health_pass_rate_live": signal_health["signal_health_pass"].mean(),
    "risk_health_pass_rate_live": risk_dashboard_live["risk_health_pass"].mean(),
    "live_drawdown": live_performance["max_drawdown"],
    "live_sharpe": live_performance["sharpe_proxy"],
    "research_sharpe": research_performance["sharpe_proxy"],
    "sharpe_degradation": live_vs_research.iloc[0]["sharpe_degradation"],
    "live_vol_multiple": live_vs_research.iloc[0]["live_vol_multiple"],
    "live_mean_ic": ic_monitor.iloc[0]["live_mean_ic"],
    "n_fail_alerts": int((alerts["severity"] == "FAIL").sum()) if len(alerts) else 0,
    "n_warn_alerts": int((alerts["severity"] == "WARN").sum()) if len(alerts) else 0,
    "estimated_cost_drag_ann_live": paper_nav_live["estimated_cost"].mean() * config.annualisation,
    "flag_data_health_pass_rate_below_95pct": bool(data_health_live["data_health_pass"].mean() < 0.95),
    "flag_signal_health_pass_rate_below_95pct": bool(signal_health["signal_health_pass"].mean() < 0.95),
    "flag_risk_health_pass_rate_below_95pct": bool(risk_dashboard_live["risk_health_pass"].mean() < 0.95),
    "flag_live_drawdown_fail": bool(live_performance["max_drawdown"] < config.max_drawdown_fail),
    "flag_live_drawdown_warn": bool(live_performance["max_drawdown"] < config.max_drawdown_warn),
    "flag_live_vol_multiple_high": bool(live_vs_research.iloc[0]["live_vol_multiple"] > config.max_live_vol_multiple_vs_research),
    "flag_live_mean_ic_negative": bool(ic_monitor.iloc[0]["live_mean_ic"] < 0),
    "flag_fail_alerts_present": bool((alerts["severity"] == "FAIL").sum() > 0) if len(alerts) else False,
    "flag_cost_drag_high": bool(paper_nav_live["estimated_cost"].mean() * config.annualisation > config.max_cost_drag_ann),
    "can_send_orders": False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_data_health_pass_rate_below_95pct",
        "flag_signal_health_pass_rate_below_95pct",
        "flag_risk_health_pass_rate_below_95pct",
        "flag_live_drawdown_fail",
        "flag_live_drawdown_warn",
        "flag_live_vol_multiple_high",
        "flag_live_mean_ic_negative",
        "flag_fail_alerts_present",
        "flag_cost_drag_high",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. no orders are generated;
2. no fills exist;
3. paper NAV equals cumulative paper returns;
4. target weights are finite;
5. live dashboard has one snapshot;
6. alerts have valid severities;
7. readiness never permits order sending;
8. saved monitoring outputs are finite.

In [ ]:
audit_rows = []

orders = pd.DataFrame(columns=["date", "asset", "side", "quantity"])
fills = pd.DataFrame(columns=["date", "asset", "quantity", "price"])

audit_rows.append({
    "check": "no_orders_generated",
    "value": int(len(orders)),
    "passed": bool(len(orders) == 0),
})

audit_rows.append({
    "check": "no_fills_generated",
    "value": int(len(fills)),
    "passed": bool(len(fills) == 0),
})

nav_recalc = (1 + paper_nav["net_return"]).cumprod()
nav_diff = float((nav_recalc - paper_nav["paper_nav"]).abs().max())
audit_rows.append({
    "check": "paper_nav_matches_cumulative_returns",
    "value": nav_diff,
    "passed": bool(nav_diff < 1e-12),
})

weights_finite = bool(np.isfinite(target_weights.fillna(0.0).to_numpy()).all())
audit_rows.append({
    "check": "target_weights_finite",
    "value": weights_finite,
    "passed": weights_finite,
})

snapshot_one_row = bool(len(dashboard_snapshot) == 1)
audit_rows.append({
    "check": "dashboard_snapshot_one_row",
    "value": len(dashboard_snapshot),
    "passed": snapshot_one_row,
})

valid_severities = {"INFO", "WARN", "FAIL"}
alerts_valid = bool(set(alerts["severity"]).issubset(valid_severities)) if len(alerts) else True
audit_rows.append({
    "check": "alert_severities_valid",
    "value": alerts_valid,
    "passed": alerts_valid,
})

readiness_blocks_orders = bool((readiness["can_send_orders"] == False).all())
audit_rows.append({
    "check": "readiness_never_permits_orders",
    "value": readiness_blocks_orders,
    "passed": readiness_blocks_orders,
})

finite_monitoring = bool(
    np.isfinite(monitoring_report_card.select_dtypes(include=[float, int]).to_numpy()).all()
)
audit_rows.append({
    "check": "monitoring_report_numeric_outputs_finite",
    "value": finite_monitoring,
    "passed": finite_monitoring,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Practical checklist for live paper trading without execution

Before moving from backtest to limited production:

1. **Run shadow mode first**  
   No orders until monitoring is stable.

2. **Track data health daily**  
   Missing, stale, and suspicious prices must be visible.

3. **Monitor signal drift**  
   Live signal distributions should resemble research.

4. **Monitor target-risk intent**  
   Gross, net, class, and single-name exposures should stay inside limits.

5. **Monitor paper performance**  
   Paper NAV should be benchmarked and drawdown-controlled.

6. **Track live-vs-research degradation**  
   Expect some decay, but measure it.

7. **Monitor IC**  
   Alpha should not immediately flip sign live.

8. **Estimate cost and capacity pressure**  
   Even without execution, planned turnover and participation matter.

9. **Create alerts**  
   Dashboard without alerting is just decoration.

10. **Block execution explicitly**  
   Paper dashboard should not have permission to send orders.

## 25. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/live_paper_trading_dashboard_without_execution_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "returns_raw": output_dir / "returns_raw.csv",
    "prices_raw": output_dir / "prices_raw.csv",
    "returns_clean": output_dir / "returns_clean.csv",
    "prices_clean": output_dir / "prices_clean.csv",
    "spread_bps": output_dir / "spread_bps.csv",
    "volume_usd": output_dir / "volume_usd.csv",
    "metadata": output_dir / "metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "factors": output_dir / "factors.csv",
    "benchmark": output_dir / "benchmark.csv",
    "data_health": output_dir / "data_health.csv",
    "signal": output_dir / "signal.csv",
    "target_weights": output_dir / "target_weights.csv",
    "signal_health": output_dir / "signal_health.csv",
    "risk_dashboard": output_dir / "risk_dashboard.csv",
    "participation": output_dir / "participation.csv",
    "class_exposure": output_dir / "class_exposure.csv",
    "paper_nav": output_dir / "paper_nav.csv",
    "performance_comparison": output_dir / "performance_comparison.csv",
    "rolling_20": output_dir / "rolling_20_monitoring.csv",
    "live_vs_research": output_dir / "live_vs_research.csv",
    "ic_series": output_dir / "ic_series.csv",
    "ic_monitor": output_dir / "ic_monitor.csv",
    "alerts": output_dir / "alerts.csv",
    "dashboard_snapshot": output_dir / "dashboard_snapshot.csv",
    "readiness": output_dir / "readiness.csv",
    "alert_summary": output_dir / "alert_summary.csv",
    "asset_attribution_live": output_dir / "asset_attribution_live.csv",
    "class_attribution_live": output_dir / "class_attribution_live.csv",
    "monitoring_report_card": output_dir / "monitoring_report_card.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "orders_empty": output_dir / "orders_empty.csv",
    "fills_empty": output_dir / "fills_empty.csv",
    "manifest": output_dir / "manifest.json",
}

returns_raw.to_csv(paths["returns_raw"])
prices_raw.to_csv(paths["prices_raw"])
returns.to_csv(paths["returns_clean"])
prices.to_csv(paths["prices_clean"])
spread_bps.to_csv(paths["spread_bps"])
volume_usd.to_csv(paths["volume_usd"])
metadata.to_csv(paths["metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
factors.to_csv(paths["factors"])
benchmark.to_frame("benchmark_return").to_csv(paths["benchmark"])
data_health.to_csv(paths["data_health"])
signal.to_csv(paths["signal"])
target_weights.to_csv(paths["target_weights"])
signal_health.to_csv(paths["signal_health"])
risk_dashboard.to_csv(paths["risk_dashboard"])
participation.to_csv(paths["participation"])
class_exposure_table.to_csv(paths["class_exposure"], index=False)
paper_nav.to_csv(paths["paper_nav"])
performance_comparison.to_csv(paths["performance_comparison"], index=False)
rolling_20.to_csv(paths["rolling_20"])
live_vs_research.to_csv(paths["live_vs_research"], index=False)
ic_series.to_csv(paths["ic_series"])
ic_monitor.to_csv(paths["ic_monitor"], index=False)
alerts.to_csv(paths["alerts"], index=False)
dashboard_snapshot.to_csv(paths["dashboard_snapshot"], index=False)
readiness.to_csv(paths["readiness"], index=False)
alert_summary.to_csv(paths["alert_summary"], index=False)
asset_attribution_live.to_csv(paths["asset_attribution_live"], index=False)
class_attribution_live.to_csv(paths["class_attribution_live"], index=False)
monitoring_report_card.to_csv(paths["monitoring_report_card"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)
orders.to_csv(paths["orders_empty"], index=False)
fills.to_csv(paths["fills_empty"], index=False)

manifest = {
    "dataset_name": "live_paper_trading_dashboard_without_execution_outputs",
    "schema_version": "live_paper_trading_dashboard_without_execution_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "assets": assets,
    "research_start": str(research_index[0]),
    "research_end": str(research_index[-1]),
    "live_start": str(live_index[0]),
    "live_end": str(live_index[-1]),
    "dashboard_snapshot": dashboard_snapshot.to_dict(orient="records"),
    "readiness": readiness.to_dict(orient="records"),
    "monitoring_report_card": monitoring_report_card.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "orders_generated": int(len(orders)),
    "fills_generated": int(len(fills)),
    "can_send_orders": False,
    "notes": [
        "This dashboard is explicitly paper/shadow mode without execution.",
        "No orders or fills are generated.",
        "Paper NAV uses lagged target weights and estimated costs.",
        "Data health, signal drift, risk exposure, performance, IC, cost and capacity are monitored.",
        "Readiness GREEN means eligible for review, not approval to trade.",
        "The dashboard stores alerts, snapshots, governance flags and audit checks.",
        "This notebook is a pre-production monitoring layer, not a trading system."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["dashboard_snapshot"], paths["readiness"], paths["monitoring_report_card"], paths["governance_flags"], paths["manifest"]

## 26. Limitations

### Synthetic live data

The live paper data is simulated. Real monitoring needs robust data vendors, exchange calendars, data timestamps, and production logging.

### No broker or OMS integration

This notebook intentionally does not connect to a broker or send orders.

### Paper NAV is approximate

Paper PnL uses target weights and next-period returns. It does not include real fill prices, partial fills, queueing, or execution algorithms.

### Cost model is estimated

Costs are only estimates based on turnover and participation.

### Alert thresholds are illustrative

Production thresholds should be set by risk, PMs, and engineering.

### No persistent database

The notebook saves CSVs, but a real system needs durable storage, run IDs, and immutable logs.

### No human approval workflow

Readiness status is generated, but actual approval must be external and documented.

## 27. Research frontier and extensions

Important extensions include:

1. automated daily report generation;
2. Slack/email alert routing;
3. dashboard web app;
4. feature-store integration;
5. model registry and config hashing;
6. paper broker integration with disabled execution;
7. live TCA prediction versus actual fills;
8. kill-switch simulation;
9. drift detectors such as PSI, KS, and Wasserstein distance;
10. Chinese futures shadow dashboard with night sessions, exchange calendars, price limits, margin estimates, and L1 bid/ask monitoring.

## 28. Suggested follow-up notebook

This naturally leads to:

**05_12_full_research_to_production_checklist**

That final Phase 5 notebook should combine:

- leakage checks;
- purged CV;
- PBO;
- White Reality Check;
- Deflated Sharpe;
- Bayesian calibration;
- turnover/capacity;
- paper trading monitoring;
- production approval gates.

## 29. Summary

This notebook implemented a live paper-trading dashboard without execution.

It showed how to:

1. simulate research and live paper data;
2. inject live data quality issues;
3. compute data health checks;
4. build live strategy signals;
5. monitor signal drift;
6. compute target weights without orders;
7. monitor target exposures and turnover;
8. estimate costs and participation;
9. compute paper NAV;
10. compare live paper performance to research baseline;
11. monitor rolling Sharpe and drawdown;
12. monitor live IC;
13. generate daily alerts;
14. build a latest dashboard snapshot;
15. produce go/no-go readiness status;
16. attribute live paper returns;
17. create monitoring report cards;
18. create governance flags and audit checks;
19. save outputs and manifests;
20. explicitly verify that no orders or fills are generated.

The key computational takeaway:

> A live paper dashboard is a stateful monitoring system for data, signal, risk and performance health, not an execution engine.

The key financial takeaway:

> Before a strategy earns the right to trade capital, it must first prove it can run cleanly in shadow mode without surprising the researcher, PM, or risk manager.

## 30. Further reading

- Institutional model-risk and production-readiness documentation.
- López de Prado, *Advances in Financial Machine Learning.*
- Grinold and Kahn, *Active Portfolio Management.*
- Literature on live model monitoring, drift detection, and shadow deployment.
- Transaction-cost-analysis literature on expected versus realised implementation quality.
- Engineering literature on observability, alerting, runbooks, and incident management.